# 090 — Generación musical y de audio

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Tres representaciones del audio**: forma de onda cruda (24 000 muestras/s a
24 kHz — secuencias enormes), espectrograma mel (imagen tiempo×frecuencia,
~100 frames/s, pierde la fase → necesita vocoder) y **tokens neuronales** de un
códec (EnCodec, SoundStream).

**Códec neuronal con RVQ**: el encoder reduce la onda a ~75 latentes/s; cada
latente se discretiza con **cuantización vectorial residual**: Q codebooks en
cascada donde cada uno cuantiza el residuo del anterior
(`z ≈ C₁[k₁] + … + C_Q[k_Q]`). Cada frame queda como Q índices de log₂(K) bits.

**Dos familias generativas**: autorregresivos sobre tokens (MusicGen, AudioLM —
un transformer predice índices RVQ condicionado por texto, con *delay pattern*
para no aplanar los Q streams) frente a difusión sobre espectrogramas mel +
vocoder (hereda la maquinaria de imagen, pero duración fija y fase perdida).


## 🧮 Ejemplo de referencia

Audio 24 kHz, códec a 75 frames/s, 8 codebooks de 1024 entradas:
`75 · 8 · log₂(1024) = 75 · 8 · 10 = 6 000 bits/s = 6 kbps`, frente a PCM
`24 000 · 16 = 384 kbps` → compresión **64×**.

Longitud de secuencia para 30 s: 2 250 frames; aplanado serían 18 000 tokens,
con delay pattern ~2 257 pasos; muestra a muestra serían 720 000 pasos. La
compresión del códec es lo que hace viable un transformer musical.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("generation", seed=90)
show(result)


## Reflexión

1. ¿Por qué la primera capa del RVQ concentra la información "gruesa" de la señal y qué implica truncar la cascada a menos codebooks para el bitrate y la calidad?
2. ¿Qué problema resuelve el delay pattern de MusicGen frente a aplanar los 8 codebooks en una sola secuencia, y cuál es su costo?
3. La difusión sobre espectrogramas mel necesita un vocoder para producir la onda: ¿por qué (qué información falta) y qué artefactos puede introducir esa etapa extra?
